# Single STAC Item: `open_item` and `open_cog`

Two single-item helpers in `lazycogs`, both lazy and read at the asset's native grid:

- `lazycogs.open_item` — stack several same-grid bands of one STAC item into a `(band, y, x)` DataArray.
- `lazycogs.open_cog` — read one Cloud-Optimized GeoTIFF at its native CRS, resolution, and shape.

First, find one low-cloud Sentinel-2 scene and configure a store for the public bucket (no credentials needed).

In [1]:
import rustac
from obstore.store import S3Store

import lazycogs

items = await rustac.search(
    href="https://earth-search.aws.element84.com/v1",
    collections=["sentinel-2-c1-l2a"],
    bbox=[4.8, 52.3, 5.0, 52.5],  # Amsterdam
    datetime="2023-06-01/2023-06-30",
    query={"eo:cloud_cover": {"lt": 10}},
    max_items=1,
)
item = items[0]

store = S3Store(
    bucket="e84-earth-search-sentinel-data",
    region="us-west-2",
    skip_signature=True,
    virtual_hosted_style_request=True,
)

## Open a Multi-Band Item with `open_item`

`lazycogs.open_item` stacks several single-band assets of one STAC item into a `(band, y, x)` DataArray at their shared native grid. Here we request the `red` and `green` bands. Notice that both of these assets have the same grid. If we were to pick bands with different grids, `lazycogs.open_item` would raise an error, `lazycogs.open` would then be the correct function to use.

Opening fetches metadata for both assets without decoding pixels. The `sel` and `isel` calls make no storage requests. Loading requests the native-resolution window from the `red` asset only; TIFF tile reads can extend beyond that window.

In [2]:
da = lazycogs.open_item(item, bands=["red", "green"], store=store)
da

<xarray.DataArray (band: 2, y: 10980, x: 10980)> Size: 482MB
[241120800 values with dtype=uint16]
Coordinates:
  * band         (band) <U5 40B 'red' 'green'
  * y            (y) float64 88kB 5.8e+06 5.8e+06 5.8e+06 ... 5.69e+06 5.69e+06
  * x            (x) float64 88kB 6e+05 6e+05 6e+05 ... 7.098e+05 7.098e+05
    spatial_ref  int64 8B 0
Indexes:
  ┌ x        RasterIndex (crs=EPSG:32631)
  └ y
Attributes:
    grid_mapping:  spatial_ref
    _FillValue:    0.0
    scale_factor:  0.0001
    add_offset:    -0.1

Load a 256 x 256 array of pixels for each band.

In [6]:
window = da.isel(y=slice(100, 356), x=slice(200, 456))
window.load()

<xarray.DataArray (band: 2, y: 256, x: 256)> Size: 262kB
array([[[3736, 3382, 2566, ..., 1195, 1318, 1354],
        [3654, 3158, 2236, ..., 1320, 1349, 1408],
        [3500, 2826, 2015, ..., 1415, 1380, 1420],
        ...,
        [2256, 2330, 2410, ..., 1954, 1578, 1416],
        [2220, 2192, 2222, ..., 1642, 1501, 1632],
        [2142, 2212, 2086, ..., 1450, 1642, 1730]],

       [[3352, 3062, 2454, ..., 1372, 1538, 1572],
        [3288, 2964, 2242, ..., 1548, 1592, 1561],
        [3208, 2654, 2013, ..., 1586, 1621, 1595],
        ...,
        [2050, 2014, 2110, ..., 1728, 1552, 1470],
        [1994, 1872, 1898, ..., 1554, 1503, 1656],
        [1886, 1924, 1872, ..., 1476, 1609, 1736]]],
      shape=(2, 256, 256), dtype=uint16)
Coordinates:
  * band         (band) <U5 40B 'red' 'green'
  * y            (y) float64 2kB 5.799e+06 5.799e+06 ... 5.796e+06 5.796e+06
  * x            (x) float64 2kB 6.02e+05 6.02e+05 ... 6.045e+05 6.046e+05
    spatial_ref  int64 8B 0
Indexes:
  ┌ x        RasterIndex (crs=EPSG:32631)
  └ y
Attributes:
    grid_mapping:  spatial_ref
    _FillValue:    0.0
    scale_factor:  0.0001
    add_offset:    -0.1

## Open a Single COG with `open_cog`

`lazycogs.open_cog` reads one Cloud-Optimized GeoTIFF in place — native CRS, resolution, and shape, with no reprojection. Here we open just the `nir` asset of the same item directly from its href.

In [7]:
nir = lazycogs.open_cog(item["assets"]["nir"]["href"], store=store)
nir

<xarray.DataArray (band: 1, y: 10980, x: 10980)> Size: 241MB
[120560400 values with dtype=uint16]
Coordinates:
  * band         (band) int64 8B 1
  * y            (y) float64 88kB 5.8e+06 5.8e+06 5.8e+06 ... 5.69e+06 5.69e+06
  * x            (x) float64 88kB 6e+05 6e+05 6e+05 ... 7.098e+05 7.098e+05
    spatial_ref  int64 8B 0
Indexes:
  ┌ x        RasterIndex (crs=EPSG:32631)
  └ y
Attributes:
    grid_mapping:  spatial_ref
    _FillValue:    0.0
    scale_factor:  0.0001
    add_offset:    -0.1

In [8]:
nir_window = nir.isel(y=slice(100, 356), x=slice(200, 456))
nir_window.load()

<xarray.DataArray (band: 1, y: 256, x: 256)> Size: 131kB
array([[[4032, 3778, 3403, ..., 2659, 3911, 4090],
        [3953, 3664, 3348, ..., 3877, 4053, 4006],
        [3844, 3492, 3396, ..., 3912, 4085, 4087],
        ...,
        [3350, 3281, 3281, ..., 3479, 3557, 3924],
        [3236, 3230, 3291, ..., 3366, 4018, 4336],
        [3208, 3260, 3310, ..., 3436, 4023, 4112]]],
      shape=(1, 256, 256), dtype=uint16)
Coordinates:
  * band         (band) int64 8B 1
  * y            (y) float64 2kB 5.799e+06 5.799e+06 ... 5.796e+06 5.796e+06
  * x            (x) float64 2kB 6.02e+05 6.02e+05 ... 6.045e+05 6.046e+05
    spatial_ref  int64 8B 0
Indexes:
  ┌ x        RasterIndex (crs=EPSG:32631)
  └ y
Attributes:
    grid_mapping:  spatial_ref
    _FillValue:    0.0
    scale_factor:  0.0001
    add_offset:    -0.1